In [1]:
import os

import psycopg
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

TABLE_NAME = "users_churn"  

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)


EXPERIMENT_NAME = "churn_troston" 
RUN_NAME = "eda"

ASSETS_DIR = "assets"


pd.options.display.max_columns = 100
pd.options.display.max_rows = 64

sns.set_style("white")
sns.set_theme(style="whitegrid")

In [2]:
with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

df.head()

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,1,7590-VHVEG,2020-01-01,NaT,Month-to-month,Yes,Electronic check,29.85,29.85,DSL,No,Yes,No,No,No,No,Female,0,Yes,No,None,0
1,2,5575-GNVDE,2017-04-01,NaT,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No,0
2,3,3668-QPYBK,2019-10-01,2019-12-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No,1
3,4,7795-CFOCW,2016-05-01,NaT,One year,No,Bank transfer (automatic),42.30,1840.75,DSL,Yes,No,Yes,Yes,No,No,Male,0,No,No,None,0
4,5,9237-HQITU,2019-09-01,2019-11-01,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,No,No,No,No,No,No,Female,0,No,No,No,1


Спринт 2/6 → Тема 3/6: Feature engineering — генерация и отбор признаков → Урок 5/11: Работа с признаками. Практика

In [3]:
import os

import pandas as pd
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, 
    SplineTransformer, 
    QuantileTransformer, 
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)

TABLE_NAME = 'users_churn'

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = 'churn_troston'
RUN_NAME = "preprocessing" 
REGISTRY_MODEL_NAME = 'Model4EncodedFeatures'

In [4]:
import os
import mlflow
import psycopg
import pandas as pd

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"), 
    "port": os.getenv("DB_DESTINATION_PORT") ,
    "dbname": os.getenv("DB_DESTINATION_NAME") ,
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD") ,
}
assert all([var_value != "" for var_value in list(postgres_credentials.values())])

connection.update(postgres_credentials)

TABLE_NAME = "users_churn"


with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]


df = pd.DataFrame(data, columns=columns).drop('end_date',axis=1)

In [8]:

df = df.dropna()
y = df['target']
dfc = pd.read_csv('clean_users_churn.csv')
dfc

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,1,7590-VHVEG,2020-01-01,NaN,Month-to-month,Yes,Electronic check,29.85,29.85,DSL,No,Yes,No,No,No,No,Female,0,Yes,No,No,0
1,2,5575-GNVDE,2017-04-01,NaN,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No,0
2,3,3668-QPYBK,2019-10-01,2019-12-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No,1
3,4,7795-CFOCW,2016-05-01,NaN,One year,No,Bank transfer (automatic),42.30,1840.75,DSL,Yes,No,Yes,Yes,No,No,Male,0,No,No,No,0
4,5,9237-HQITU,2019-09-01,2019-11-01,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,No,No,No,No,No,No,Female,0,No,No,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,7039,6840-RESVB,2018-02-01,NaN,One year,Yes,Mailed check,84.80,1990.50,DSL,Yes,No,Yes,Yes,Yes,Yes,Male,0,Yes,Yes,Yes,0
7039,7040,2234-XADUH,2014-02-01,NaN,One year,Yes,Credit card (automatic),103.20,7362.90,Fiber optic,No,Yes,Yes,No,Yes,Yes,Female,0,Yes,Yes,Yes,0
7040,7041,4801-JZAZL,2019-03-01,NaN,Month-to-month,Yes,Electronic check,29.60,346.45,DSL,Yes,No,No,No,No,No,Female,0,Yes,Yes,No,0
7041,7042,8361-LTMKD,2019-07-01,2019-11-01,Month-to-month,Yes,Mailed check,74.40,306.60,Fiber optic,No,No,No,No,No,No,Male,1,Yes,No,Yes,1


In [9]:
dfc.isna().sum()

id                      0
customer_id             0
begin_date              0
end_date             5174
type                    0
paperless_billing       0
payment_method          0
monthly_charges         0
total_charges           0
internet_service        0
online_security         0
online_backup           0
device_protection       0
tech_support            0
streaming_tv            0
streaming_movies        0
gender                  0
senior_citizen          0
partner                 0
dependents              0
multiple_lines          0
target                  0
dtype: int64

In [135]:
# obj_df = df.select_dtypes(include="object").drop('customer_id',axis=1)
# num_df = df.select_dtypes(include=["float64"])


cat_columns = ["type", "payment_method", "internet_service", "gender", 'paperless_billing', 'online_security', 'online_backup',
               'device_protection',	'tech_support',	'streaming_tv',	'streaming_movies', 'partner',	'dependents',	'multiple_lines','senior_citizen' ]

num_columns = df.drop(cat_columns, axis=1)
# num_columns['begin_date'] = pd.to_datetime(num_columns['begin_date'])
# num_columns['end_date'] = pd.to_datetime(num_columns['end_date'])
# num_columns['n_days'] = (num_columns['end_date'] - num_columns['begin_date']).dt.days
num_columns = num_columns.drop(columns=['begin_date','target','id', 'customer_id'])


encoder_oh = OneHotEncoder(
    categories="auto",
    handle_unknown="ignore",
    max_categories=10,
    sparse_output=False,
    drop="first"
)

encoded_features = encoder_oh.fit_transform(df[cat_columns].to_numpy())

encoded_cat_df = pd.DataFrame(
    encoded_features,
    columns=encoder_oh.get_feature_names_out(cat_columns))

# obj_df = pd.concat([obj_df, encoded_df], axis=1)
encoded_cat_df


,type_One year,type_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,internet_service_Fiber optic,gender_Male,paperless_billing_Yes,online_security_Yes,online_backup_Yes,device_protection_Yes,tech_support_Yes,streaming_tv_Yes,streaming_movies_Yes,partner_Yes,dependents_Yes,multiple_lines_Yes,senior_citizen_1
0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
4,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4827,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4828,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
4829,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0
4830,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0


In [136]:
y = df['target']
# df['begin_date'] = pd.to_datetime(df['begin_date'])
# df['end_date'] = pd.to_datetime(df['end_date'])
# df['n_days'] = (df['end_date'] - df['begin_date']).dt.days
df.drop(columns=['target','id', 'customer_id'], inplace=True)
df


,begin_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines
1,2017-04-01,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No
2,2019-10-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No
4,2019-09-01,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,No,No,No,No,No,No,Female,0,No,No,No
5,2019-03-01,Month-to-month,Yes,Electronic check,99.65,820.50,Fiber optic,No,No,Yes,No,Yes,Yes,Female,0,No,No,Yes
6,2018-04-01,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,Fiber optic,No,Yes,No,No,Yes,No,Male,0,No,Yes,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7035,2018-07-01,Month-to-month,Yes,Bank transfer (automatic),78.70,1495.10,Fiber optic,No,No,No,No,Yes,No,Male,0,No,No,No
7038,2018-02-01,One year,Yes,Mailed check,84.80,1990.50,DSL,Yes,No,Yes,Yes,Yes,Yes,Male,0,Yes,Yes,Yes
7039,2014-02-01,One year,Yes,Credit card (automatic),103.20,7362.90,Fiber optic,No,Yes,Yes,No,Yes,Yes,Female,0,Yes,Yes,Yes
7041,2019-07-01,Month-to-month,Yes,Mailed check,74.40,306.60,Fiber optic,No,No,No,No,No,No,Male,1,Yes,No,Yes


In [137]:
num_columns

,monthly_charges,total_charges
1,56.95,1889.50
2,53.85,108.15
4,70.70,151.65
5,99.65,820.50
6,89.10,1949.40
...,...,...
7035,78.70,1495.10
7038,84.80,1990.50
7039,103.20,7362.90
7041,74.40,306.60


In [138]:
numeric_columns = num_columns.columns.tolist()
n_knots = 3
degree_spline = 4
n_quantiles = 100
degree = 3
n_bins = 5
encode = 'ordinal'
strategy = 'uniform'
subsample = None



# RobustScaler
encoder_rb = RobustScaler()
encoded_features = encoder_rb.fit_transform(num_columns.to_numpy())
encoded_df = pd.DataFrame(encoded_features, columns=encoder_rb.get_feature_names_out(numeric_columns))
encoded_df.columns = [col + f"_robust" for col in num_columns]
num_df = encoded_df

# PolynomialFeatures
encoder_pol = PolynomialFeatures(degree=degree)
encoded_features = encoder_pol.fit_transform(num_columns.to_numpy())
encoded_df = pd.DataFrame(encoded_features, columns=encoder_pol.get_feature_names_out(numeric_columns))
encoded_df = encoded_df.iloc[:, 1 + len(numeric_columns):]
encoded_df.columns = [col + f"_poly" for col in encoded_df.columns]
num_df = pd.concat([num_df, encoded_df], axis=1)





In [139]:
num_df

,monthly_charges_robust,total_charges_robust,monthly_charges^2_poly,monthly_charges total_charges_poly,total_charges^2_poly,monthly_charges^3_poly,monthly_charges^2 total_charges_poly,monthly_charges total_charges^2_poly,total_charges^3_poly
0,-0.986011,-0.109491,3243.3025,107607.0250,3.570210e+06,1.847061e+05,6.128220e+06,2.033235e+08,6.745912e+09
1,-1.105644,-0.532458,2899.8225,5823.8775,1.169642e+04,1.561554e+05,3.136158e+05,6.298524e+05,1.264968e+06
2,-0.455379,-0.522130,4998.4900,10721.6550,2.299772e+04,3.533932e+05,7.580210e+05,1.625939e+06,3.487605e+06
3,0.661843,-0.363316,9930.1225,81762.8250,6.732202e+05,9.895367e+05,8.147666e+06,6.708640e+07,5.523772e+08
4,0.254703,-0.095268,7938.8100,173691.5400,3.800160e+06,7.073480e+05,1.547592e+07,3.385943e+08,7.408033e+09
...,...,...,...,...,...,...,...,...,...
4827,-0.146647,-0.203138,6193.6900,117664.3700,2.235324e+06,4.874434e+05,9.260186e+06,1.759200e+08,3.342033e+09
4828,0.088760,-0.085509,7191.0400,168794.4000,3.962090e+06,6.098002e+05,1.431377e+07,3.359853e+08,7.886541e+09
4829,0.798842,1.190126,10650.2400,759851.2800,5.421230e+07,1.099105e+06,7.841665e+07,5.594709e+09,3.991597e+11
4830,-0.312590,-0.485338,5535.3600,22811.0400,9.400356e+04,4.118308e+05,1.697141e+06,6.993865e+06,2.882149e+07


In [140]:
df

,begin_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines
1,2017-04-01,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No
2,2019-10-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No
4,2019-09-01,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,No,No,No,No,No,No,Female,0,No,No,No
5,2019-03-01,Month-to-month,Yes,Electronic check,99.65,820.50,Fiber optic,No,No,Yes,No,Yes,Yes,Female,0,No,No,Yes
6,2018-04-01,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,Fiber optic,No,Yes,No,No,Yes,No,Male,0,No,Yes,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7035,2018-07-01,Month-to-month,Yes,Bank transfer (automatic),78.70,1495.10,Fiber optic,No,No,No,No,Yes,No,Male,0,No,No,No
7038,2018-02-01,One year,Yes,Mailed check,84.80,1990.50,DSL,Yes,No,Yes,Yes,Yes,Yes,Male,0,Yes,Yes,Yes
7039,2014-02-01,One year,Yes,Credit card (automatic),103.20,7362.90,Fiber optic,No,Yes,Yes,No,Yes,Yes,Female,0,Yes,Yes,Yes
7041,2019-07-01,Month-to-month,Yes,Mailed check,74.40,306.60,Fiber optic,No,No,No,No,No,No,Male,1,Yes,No,Yes


In [141]:
num_columns

,monthly_charges,total_charges
1,56.95,1889.50
2,53.85,108.15
4,70.70,151.65
5,99.65,820.50
6,89.10,1949.40
...,...,...
7035,78.70,1495.10
7038,84.80,1990.50
7039,103.20,7362.90
7041,74.40,306.60


In [142]:
cat_columns

['type',
 'payment_method',
 'internet_service',
 'gender',
 'paperless_billing',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'partner',
 'dependents',
 'multiple_lines',
 'senior_citizen']

In [143]:
numeric_columns_list = num_columns.columns.tolist()

numeric_transformer = ColumnTransformer(transformers=[ ('rb', encoder_rb, numeric_columns_list), ('pol', encoder_pol, numeric_columns_list)])

categorical_transformer = Pipeline(steps=[('encoder', encoder_oh)])

preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_columns_list), ('cat', categorical_transformer, cat_columns)], n_jobs=-1)

encoded_features = preprocessor.fit_transform(df)

transformed_df = pd.DataFrame(
    encoded_features,
    columns=preprocessor.get_feature_names_out()
)

transformed_df

,num__rb__monthly_charges,num__rb__total_charges,num__pol__1,num__pol__monthly_charges,num__pol__total_charges,num__pol__monthly_charges^2,num__pol__monthly_charges total_charges,num__pol__total_charges^2,num__pol__monthly_charges^3,num__pol__monthly_charges^2 total_charges,num__pol__monthly_charges total_charges^2,num__pol__total_charges^3,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__gender_Male,cat__paperless_billing_Yes,cat__online_security_Yes,cat__online_backup_Yes,cat__device_protection_Yes,cat__tech_support_Yes,cat__streaming_tv_Yes,cat__streaming_movies_Yes,cat__partner_Yes,cat__dependents_Yes,cat__multiple_lines_Yes,cat__senior_citizen_1
0,-0.986011,-0.109491,1.0,56.95,1889.50,3243.3025,107607.0250,3.570210e+06,1.847061e+05,6.128220e+06,2.033235e+08,6.745912e+09,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.105644,-0.532458,1.0,53.85,108.15,2899.8225,5823.8775,1.169642e+04,1.561554e+05,3.136158e+05,6.298524e+05,1.264968e+06,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.455379,-0.522130,1.0,70.70,151.65,4998.4900,10721.6550,2.299772e+04,3.533932e+05,7.580210e+05,1.625939e+06,3.487605e+06,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.661843,-0.363316,1.0,99.65,820.50,9930.1225,81762.8250,6.732202e+05,9.895367e+05,8.147666e+06,6.708640e+07,5.523772e+08,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
4,0.254703,-0.095268,1.0,89.10,1949.40,7938.8100,173691.5400,3.800160e+06,7.073480e+05,1.547592e+07,3.385943e+08,7.408033e+09,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4827,-0.146647,-0.203138,1.0,78.70,1495.10,6193.6900,117664.3700,2.235324e+06,4.874434e+05,9.260186e+06,1.759200e+08,3.342033e+09,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4828,0.088760,-0.085509,1.0,84.80,1990.50,7191.0400,168794.4000,3.962090e+06,6.098002e+05,1.431377e+07,3.359853e+08,7.886541e+09,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
4829,0.798842,1.190126,1.0,103.20,7362.90,10650.2400,759851.2800,5.421230e+07,1.099105e+06,7.841665e+07,5.594709e+09,3.991597e+11,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0
4830,-0.312590,-0.485338,1.0,74.40,306.60,5535.3600,22811.0400,9.400356e+04,4.118308e+05,1.697141e+06,6.993865e+06,2.882149e+07,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0


In [144]:
preprocessor

ColumnTransformer(n_jobs=-1,
                  transformers=[('num',
                                 ColumnTransformer(transformers=[('rb',
                                                                  RobustScaler(),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('pol',
                                                                  PolynomialFeatures(degree=3),
                                                                  ['monthly_charges',
                                                                   'total_charges'])]),
                                 ['monthly_charges', 'total_charges']),
                                ('cat',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                max_categories=10,
                                                                sparse_output=False))]),
                                 ['type', 'payment_method', 'internet_service',
                                  'gender', 'paperless_billing',
                                  'online_security', 'online_backup',
                                  'device_protection', 'tech_support',
                                  'streaming_tv', 'streaming_movies', 'partner',
                                  'dependents', 'multiple_lines',
                                  'senior_citizen'])])

In [145]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.sklearn.log_model(preprocessor, "column_transformer") 

2026/04/07 13:54:27 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


In [146]:
run_id

'b17f521acdf741fa865870e8d3ab19a7'

## Задание 5. Обучим новую версию модели на преобразованных признаках и зарегистрируем ее в mlflow

In [148]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

In [156]:
X = transformed_df
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.8, random_state=42)

In [164]:
model = CatBoostClassifier( iterations=5000, early_stopping_rounds=100, eval_metric='Accuracy').fit(X_train, y_train, cat_features=cat_cols, eval_set=(X_test, y_test), verbose=100)
y_pred = model.predict(X_test)
prediction = model.predict(X_train)
y_prob = model.predict_proba(X_test)[:,1]

Learning rate set to 0.021973
0:	learn: 0.7518758	test: 0.7373320	best: 0.7373320 (0)	total: 4.98ms	remaining: 24.9s
100:	learn: 0.7746442	test: 0.7580145	best: 0.7631851 (63)	total: 474ms	remaining: 23s
200:	learn: 0.7883571	test: 0.7621510	best: 0.7673216 (182)	total: 916ms	remaining: 21.9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7673216132
bestIteration = 182

Shrink model to first 183 iterations.


In [165]:
from sklearn.metrics import accuracy_score, roc_auc_score
from mlflow.models import infer_signature

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.log_params(model.get_params())

    mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test)))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

    signature = infer_signature(X_train, model.predict(X_train))

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="catboost_model",
        signature=signature,
        registered_model_name=REGISTRY_MODEL_NAME 
    )

print(f"Run ID: {run_id}")
print(f"Model URI: {model_info.model_uri}")

Successfully registered model 'Model4EncodedFeatures'.
2026/04/07 14:04:52 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: Model4EncodedFeatures, version 1


Run ID: be4929e61f6b4021ba4141797450d8b3
Model URI: runs:/be4929e61f6b4021ba4141797450d8b3/catboost_model


Created version '1' of model 'Model4EncodedFeatures'.
